# aiLand v0 — exploring the XGBoost emulator of ecLand

The training, inference and evaluation logic lives in the `ailand` package
(`src/ailand/`), not in this notebook. Run it from the command line:

```bash
python -m ailand.train    --preset v1
python -m ailand.infer    --preset v1 --point 5
python -m ailand.evaluate --preset v1
```

This notebook is the interactive front-end to the same code — for looking at the
data, comparing variable-set presets and plotting. The original ec-land-db
notebook is preserved verbatim at
`notebooks/upstream/train_ai_land_example_v0_pristine.ipynb`.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "src" / "ailand").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import matplotlib.pyplot as plt

from ailand import config, data, train, infer, evaluate

print("presets:", sorted(config.PRESETS))
print("store:", config.DATA)

## The data

The mock store holds 10 land points at 6-hourly resolution for 2020–2022,
with 37 variables. For context, the real aiLand v1 training data is 171,039
land points over 1998–2024 — see `data/EXTERNAL.md` for where it lives on `/lus`.

In [ ]:
ds = data.open_store()
ds

## Variable sets

`prognostic` variables are predicted as 6-hourly increments and fed back into the
input vector at the next step. `diagnostic` variables are predicted as absolute
values and never fed back.

In [ ]:
for name in sorted(config.PRESETS):
    prog, diag, feat = config.resolve(name)
    print(f"{name:9s} {len(feat):2d} features | prognostic: {', '.join(prog)}")
    print(f"{'':9s} {'':2s}          | diagnostic: {', '.join(diag) or '-'}")

## Why the increments need scaling

Increment magnitudes span four orders of magnitude. With a single summed-squared-error
objective, the loss is driven almost entirely by soil temperature and snow, and the
remaining variables contribute almost no gradient. This is what aiLand v1's
*tendency scalers* fix — pass `--scale-targets` to `ailand.train`.

In [ ]:
X, y_prog, y_diag, meta = data.training_arrays(preset="v0+snow")
scalers = data.tendency_scalers(y_prog)
for v, s in zip(meta["prognostic"], scalers):
    print(f"{v:8s} increment std = {s:12.5g}")
print(f"\nratio largest/smallest = {scalers.max() / scalers.min():.4g}")

## Train, roll out and score

Equivalent to the three command-line steps above. Reduce `n_estimators` for a
quicker pass.

In [ ]:
PRESET = "v1"

train.main(["--preset", PRESET, "--n-estimators", "1000", "--quiet"])

In [ ]:
rows = evaluate.main(["--preset", PRESET, "--quiet"])

## Comparing presets

The v0 state vector carries surface and subsurface runoff as prognostic variables.
Runoff is a flux, not a state: its rollout is essentially noise, and because it also
sits in the *input* vector that noise propagates into everything downstream. Dropping
it — which is what aiLand v1 does — is worth more than any amount of extra training.

In [ ]:
results = {}
for name in ["v0", "v0+snow", "v1"]:
    train.main(["--preset", name, "--n-estimators", "1000", "--quiet"])
    results[name] = evaluate.main(["--preset", name, "--quiet", "--no-plot"])

common = ["swvl1", "swvl2", "swvl3", "stl1", "stl2", "stl3", "snowc"]
print(f"\n{'variable':10s}" + "".join(f"{n:>12s}" for n in results))
print("-" * (10 + 12 * len(results)) + "   held-out 2022 R2")
for v in common:
    line = f"{v:10s}"
    for name, rows in results.items():
        r2 = [r["r2"] for r in rows if r["variable"] == v and r["period"] == "test"]
        line += f"{r2[0]:12.3f}" if r2 else f"{'-':>12s}"
    print(line)

## The plot

The v0 notebook's figure is convincing because the plotted range is dominated by
the seasonal cycle. Always read it next to the scores above.

In [ ]:
model, model_diag, mmeta = infer.load(PRESET.replace("+", "_"))
feats_arr, times, truth, rmeta = data.rollout_inputs(preset=PRESET, point=5)
m = {**mmeta, **rmeta}
feats_arr, diag_arr = infer.rollout(model, model_diag, m, feats_arr, verbose=False)
pred = infer.to_dataset(feats_arr, diag_arr, times, m)

evaluate.plot(pred, truth, m)